# URLFusionNet — IEEE Evidence-Locked Gate Stabilization

**Purpose:** Diagnose and address token-dominant adaptive-gate collapse **without altering the dataset, split, encoders, class weights, or primary selection metric**. This notebook is a *new validation-selected stabilization experiment*, not a claim of a completed experiment.

**Observed prior evidence (RTX 3070 Ti, TensorFlow 2.20; already executed):** The original adaptive gate obtained test Macro-F1 of **0.976385, 0.932911, 0.974994** for seeds **42, 123, 2025**. Fixed-average fusion obtained **0.974763, 0.975342, 0.976153**. Checkpoint diagnosis found almost completely token-dominant gate allocation for adaptive seed **123**. These are **historic test results**, not new results produced by this notebook.

**Controlled interventions:** (A) zero-initialized output logits for initially balanced attention, and (B) the same balanced initialization plus a weak KL penalty against degenerate gate probabilities. Both preserve an *input-dependent* gate. No hard-coded 0.5/0.5 fusion is introduced in the proposed versions.

**Leakage safeguards:** Choose between candidate configurations using **validation Macro-F1 across three predetermined seeds**. Do not evaluate candidate models on test. Only **after** selection, restore best validation checkpoints and evaluate the selected configuration once on test. The existing test set was previously used during architecture iteration, so results are **not an independent untouched final holdout**; an external temporal dataset is recommended for the paper.

**Runtime:** 6 training runs (2 candidates × 3 seeds). Separate checkpoints allow interrupted runs to resume. All results generated after execution are real; no new scores are prefilled.

## 1. Environment and experiment settings

In [ ]:
import os, gc, glob, hashlib, json, platform, random, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             f1_score, matthews_corrcoef, confusion_matrix,
                             classification_report)
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL','2')
SPLIT_SEED=42
SEEDS=[42,123,2025]
MAIN_EPOCHS=6
BATCH_SIZE=256
LEARNING_RATE=8e-4
NUM_CLASSES=4
MAX_CHAR_LEN=200
MAX_TOKEN_LEN=40
MAX_CHAR_VOCAB=256
MAX_TOKEN_VOCAB=30000
CHAR_EMB_DIM=32
TOKEN_EMB_DIM=48
CONV_FILTERS=48
LATENT_DIM=96
ATTN_HEADS=2
DROPOUT=0.20
CLASS_NAMES=['benign','defacement','phishing','malware']
LABEL_TO_ID={name:i for i,name in enumerate(CLASS_NAMES)}
ID_TO_LABEL={i:name for name,i in LABEL_TO_ID.items()}

# Choose a single output root for all new files. The WSL path maps to Windows D:\.
RESULT_DIR=Path(os.environ.get('URLFUSIONNET_RESULT_DIR', str(Path.cwd() / 'outputs' / 'stabilization'))).expanduser().resolve()
RESULT_DIR.mkdir(parents=True,exist_ok=True)
# Change to True to rerun a completed candidate; default safely resumes.
FORCE_RETRAIN=False
# A dry, quick shape/finite-value model smoke-test before the six long fits.
RUN_MODEL_SMOKE_TEST=True
CANDIDATES={
    'balanced_init': 0.0,
    'balanced_kl005': 0.005,
}

for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g,True)
    except RuntimeError: pass
GPUS=tf.config.list_physical_devices('GPU')
print('Python:',sys.version.split()[0],'TF:',tf.__version__,'scikit-learn:',sklearn.__version__)
print('GPU:',GPUS,'BATCH_SIZE:',BATCH_SIZE,'RUNS:',len(CANDIDATES)*len(SEEDS))
print('Output folder:',RESULT_DIR)

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_all_seeds(42)

## 2. Repeat exactly the original URL-level audit and 80:10:10 split

In [ ]:

KAGGLE_DATASET="sid321axn/malicious-urls-dataset"
EXPECTED_COUNTS={
    "benign":428103,
    "defacement":96457,
    "phishing":94111,
    "malware":32520,
}

candidates=[
    "/kaggle/input/malicious-urls-dataset/malicious_phish.csv",
    "./malicious_phish.csv",
    "data/raw/malicious_phish.csv",
    "../data/raw/malicious_phish.csv",
    "malicious_phish.csv",
]
manual_dataset=os.environ.get('URLFUSIONNET_DATASET_CSV')
if manual_dataset and not Path(manual_dataset).is_file():
    raise FileNotFoundError('URLFUSIONNET_DATASET_CSV does not point to a file')
csv_path=manual_dataset or next((p for p in candidates if os.path.exists(p)),None)

if csv_path is None:
    import kagglehub
    root=kagglehub.dataset_download(KAGGLE_DATASET)
    matches=glob.glob(os.path.join(root,"**","malicious_phish.csv"),recursive=True)
    if not matches:
        matches=glob.glob(os.path.join(root,"**","*.csv"),recursive=True)
    if not matches: raise FileNotFoundError("Dataset CSV not found.")
    csv_path=matches[0]

raw=pd.read_csv(csv_path)
cols={c.lower().strip():c for c in raw.columns}
url_col=next((cols[k] for k in ["url","urls","link"] if k in cols),None)
label_col=next((cols[k] for k in ["type","label","class","category"] if k in cols),None)

df=raw[[url_col,label_col]].copy()
df.columns=["url","label"]
df["url"]=df["url"].astype("string")
df["label"]=df["label"].astype("string").str.strip().str.lower()

observed={k:int(df["label"].value_counts().get(k,0)) for k in EXPECTED_COUNTS}
assert observed==EXPECTED_COUNTS, observed

df=df.dropna(subset=["url","label"]).copy()
df["url"]=df["url"].str.strip()
df=df[df["url"].str.len()>0]
df=df[df["label"].isin(CLASS_NAMES)].copy()

card=df.groupby("url",sort=False)["label"].nunique()
conflict_urls=card[card>1].index
df=df[~df["url"].isin(conflict_urls)].copy()
df=df.drop_duplicates(subset=["url","label"],keep="first").copy()
df["label_id"]=df["label"].map(LABEL_TO_ID).astype("int32")

train_df,temp_df=train_test_split(
    df,test_size=0.20,random_state=SPLIT_SEED,stratify=df["label_id"]
)
val_df,test_df=train_test_split(
    temp_df,test_size=0.50,random_state=SPLIT_SEED,stratify=temp_df["label_id"]
)
for x in [train_df,val_df,test_df]: x.reset_index(drop=True,inplace=True)

assert len(set(train_df.url)&set(val_df.url))==0
assert len(set(train_df.url)&set(test_df.url))==0
assert len(set(val_df.url)&set(test_df.url))==0

counts=train_df["label_id"].value_counts().sort_index()
CLASS_WEIGHT={
    k:float(len(train_df)/(NUM_CLASSES*counts.loc[k]))
    for k in range(NUM_CLASSES)
}

print("Clean:",len(df))
print("Train/Val/Test:",len(train_df),len(val_df),len(test_df))
print("Class weights:",CLASS_WEIGHT)


## 3. Adapt character/token vocabularies using training URLs only

In [ ]:

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
def url_token_standardize(x):
    x=tf.strings.lower(x)
    x=tf.strings.regex_replace(x,r"[^a-z0-9]+"," ")
    return tf.strings.strip(x)

char_vectorizer=layers.TextVectorization(
    max_tokens=MAX_CHAR_VOCAB,standardize=None,split="character",
    output_mode="int",output_sequence_length=MAX_CHAR_LEN
)
token_vectorizer=layers.TextVectorization(
    max_tokens=MAX_TOKEN_VOCAB,standardize=url_token_standardize,
    split="whitespace",output_mode="int",
    output_sequence_length=MAX_TOKEN_LEN
)

adapt_ds=tf.data.Dataset.from_tensor_slices(
    train_df["url"].astype(str).values
).batch(4096)
char_vectorizer.adapt(adapt_ds)
token_vectorizer.adapt(adapt_ds)

AUTOTUNE=tf.data.AUTOTUNE

def make_dataset(frame,training=False,seed=42):
    ds=tf.data.Dataset.from_tensor_slices((
        frame["url"].astype(str).values,
        frame["label_id"].astype("int32").values
    ))
    if training:
        ds=ds.shuffle(min(len(frame),200000),seed=seed,reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def make_predict_dataset(urls):
    return tf.data.Dataset.from_tensor_slices(
        np.asarray(urls,dtype=str)
    ).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds=make_dataset(val_df,False)


## 4. Reuse the same stable layers and unweighted validation Macro-F1 metric

In [ ]:

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class PositionalEmbedding(layers.Layer):
    def __init__(self,max_length,dim,**kwargs):
        super().__init__(**kwargs); self.max_length=max_length; self.dim=dim
    def build(self,input_shape):
        self.positional=self.add_weight(
            name="positional_embedding",
            shape=(1,self.max_length,self.dim),
            initializer="random_normal",trainable=True
        )
    def call(self,x):
        return x+self.positional[:,:tf.shape(x)[1],:]

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class AbsDifference(layers.Layer):
    def call(self,inputs):
        a,b=inputs; return tf.abs(a-b)

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class AdaptiveWeightedFusion(layers.Layer):
    def call(self,inputs):
        zc,zt,a=inputs
        return a[:,0:1]*zc+a[:,1:2]*zt

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class SafeMaskedMean(layers.Layer):
    def call(self,inputs):
        x,m=inputs
        m=tf.cast(m,x.dtype)[...,None]
        return tf.reduce_sum(x*m,axis=1)/tf.maximum(
            tf.reduce_sum(m,axis=1),tf.cast(1.0,x.dtype)
        )

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class SparseMacroF1(tf.keras.metrics.Metric):
    def __init__(self,num_classes=4,name="macro_f1",**kwargs):
        super().__init__(name=name,**kwargs)
        self.num_classes=num_classes
        self.cm=self.add_weight(
            name="cm",shape=(num_classes,num_classes),
            initializer="zeros",dtype=tf.float32
        )
    def update_state(self,y_true,y_pred,sample_weight=None):
        yt=tf.cast(tf.reshape(y_true,[-1]),tf.int32)
        yp=tf.argmax(y_pred,axis=-1,output_type=tf.int32)
        self.cm.assign_add(tf.math.confusion_matrix(
            yt,yp,num_classes=self.num_classes,dtype=tf.float32
        ))
    def result(self):
        tp=tf.linalg.diag_part(self.cm)
        fp=tf.reduce_sum(self.cm,axis=0)-tp
        fn=tf.reduce_sum(self.cm,axis=1)-tp
        p=tf.math.divide_no_nan(tp,tp+fp)
        r=tf.math.divide_no_nan(tp,tp+fn)
        f=tf.math.divide_no_nan(2*p*r,p+r)
        return tf.reduce_mean(f)
    def reset_state(self):
        self.cm.assign(tf.zeros_like(self.cm))


## 5. Controlled gate stabilization

The feature extractors, latent projection, residual classifier, optimizer, class weighting, number of epochs, and data order policy are identical to the completed experiments. Only the **gate output-layer initialization** and optional training-only penalty differ.

For input \(s_i=[z_i^c\Vert z_i^t]\), the gate is
\[
\alpha_i=\operatorname{softmax}(W_g\,\operatorname{GELU}(W_s s_i+b_s)+b_g),\qquad
f_i=\alpha_{i,c}z_i^c+\alpha_{i,t}z_i^t.
\]

The stabilizing term is \(\lambda\,\mathbb E_i[\mathrm{KL}(\alpha_i\Vert(0.5,0.5))]\). Its coefficient is **0** for balanced initialization alone or **0.005** for the regularized arm. This loss promotes entropy but **does not guarantee input-dependent allocation**, so entropy and across-URL gate standard deviation are measured explicitly.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class GateEntropyPenalty(layers.Layer):
    """Adds mean KL(alpha || [0.5,0.5]) during fitting; inference unchanged."""
    def __init__(self, coefficient=0.005, **kwargs):
        super().__init__(**kwargs)
        self.coefficient = float(coefficient)
    def call(self, probabilities):
        p=tf.clip_by_value(tf.cast(probabilities,tf.float32),1e-7,1.0)
        kl=tf.reduce_sum(p*tf.math.log(2.0*p),axis=-1)
        self.add_loss(self.coefficient*tf.reduce_mean(kl))
        return probabilities
    def get_config(self):
        config=super().get_config()
        config.update({'coefficient':self.coefficient})
        return config


def build_model(variant="proposed"):
    url=keras.Input(shape=(),dtype=tf.string,name="url")

    # character branch
    cids=char_vectorizer(url)
    xc=layers.Embedding(
        len(char_vectorizer.get_vocabulary()),CHAR_EMB_DIM,name="char_embedding"
    )(cids)
    pools=[]
    for k in (3,5,7):
        h=layers.Conv1D(CONV_FILTERS,k,padding="same",use_bias=False,
                        name=f"char_conv_k{k}")(xc)
        h=layers.BatchNormalization(name=f"char_bn_k{k}")(h)
        h=layers.Activation(tf.nn.gelu,name=f"char_gelu_k{k}")(h)
        pools.append(layers.GlobalMaxPooling1D(name=f"char_gmp_k{k}")(h))
    hc=layers.Concatenate(name="char_multiscale_concat")(pools)
    hc=layers.Dense(LATENT_DIM,name="char_scale_projection")(hc)

    # token branch
    tids=token_vectorizer(url)
    token_mask=layers.Lambda(
        lambda ids:tf.not_equal(ids,0),name="token_valid_mask"
    )(tids)
    xt=layers.Embedding(
        len(token_vectorizer.get_vocabulary()),TOKEN_EMB_DIM,
        mask_zero=False,name="token_embedding"
    )(tids)
    xt=PositionalEmbedding(MAX_TOKEN_LEN,TOKEN_EMB_DIM,name="token_position")(xt)
    key_mask=layers.Lambda(
        lambda m:tf.expand_dims(m,axis=1),name="token_key_mask"
    )(token_mask)
    attn=layers.MultiHeadAttention(
        num_heads=ATTN_HEADS,key_dim=TOKEN_EMB_DIM//ATTN_HEADS,
        dropout=0.10,name="token_self_attention"
    )(xt,xt,attention_mask=key_mask)
    xt=layers.LayerNormalization(name="token_attn_ln")(layers.Add()([xt,attn]))
    ff=layers.Dense(TOKEN_EMB_DIM*2,activation=tf.nn.gelu,name="token_ff1")(xt)
    ff=layers.Dropout(DROPOUT)(ff)
    ff=layers.Dense(TOKEN_EMB_DIM,name="token_ff2")(ff)
    xt=layers.LayerNormalization(name="token_ff_ln")(layers.Add()([xt,ff]))
    ht=SafeMaskedMean(name="token_safe_pool")([xt,token_mask])

    # common space
    zc=layers.LayerNormalization(name="z_char")(
        layers.Dense(LATENT_DIM,name="char_common_projection")(hc)
    )
    zt=layers.LayerNormalization(name="z_token")(
        layers.Dense(LATENT_DIM,name="token_common_projection")(ht)
    )

    if variant=="char_only":
        fused=zc
    elif variant=="token_only":
        fused=zt
    elif variant=="fixed_avg":
        fused=layers.Average(name="fixed_average_fusion")([zc,zt])
    else:
        if variant=="explicit_interaction":
            prod=layers.Multiply(name="cross_product")([zc,zt])
            diff=AbsDifference(name="cross_absdiff")([zc,zt])
            gate_input=layers.Concatenate(name="interaction_representation")(
                [zc,zt,prod,diff]
            )
        else:
            gate_input=layers.Concatenate(name="adaptive_gate_context")([zc,zt])

        gh=layers.Dense(
            LATENT_DIM//2,activation=tf.nn.gelu,name="gate_hidden"
        )(gate_input)
        init_kwargs = {}
        if variant in ("balanced_init", "balanced_kl005"):
            init_kwargs = {
                "kernel_initializer": "zeros",
                "bias_initializer": "zeros"
            }
        alpha=layers.Dense(
            2,activation="softmax",dtype="float32",name="fusion_weights",
            **init_kwargs
        )(gh)
        if variant == "balanced_kl005":
            alpha=GateEntropyPenalty(
                coefficient=0.005,name="gate_entropy_penalty"
            )(alpha)
        fused=AdaptiveWeightedFusion(name="adaptive_weighted_fusion")(
            [zc,zt,alpha]
        )

    use_residual = variant not in (
        "char_only","token_only","proposed_no_residual"
    )
    if use_residual:
        r=layers.Dense(LATENT_DIM*2,activation=tf.nn.gelu,name="residual_fc1")(fused)
        r=layers.Dropout(DROPOUT)(r)
        r=layers.Dense(LATENT_DIM,name="residual_fc2")(r)
        fused=layers.LayerNormalization(name="residual_ln")(layers.Add()([fused,r]))

    out=layers.Dense(
        NUM_CLASSES,activation="softmax",dtype="float32",name="classifier"
    )(fused)
    return keras.Model(url,out,name=f"URLFusionNet_{variant}")

def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=LEARNING_RATE,weight_decay=1e-4,clipnorm=1.0
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            SparseMacroF1(NUM_CLASSES,name="macro_f1")
        ]
    )
    return model

## 6. Non-training smoke test: finite output, proper output shapes, balanced initial gate

In [ ]:
if RUN_MODEL_SMOKE_TEST:
    for variant in CANDIDATES:
        tf.keras.backend.clear_session()
        set_all_seeds(42)
        m=compile_model(build_model(variant))
        examples=train_df['url'].iloc[:8].astype(str).values
        y=m.predict(make_predict_dataset(examples),verbose=0)
        assert y.shape==(8,4) and np.isfinite(y).all()
        gate=keras.Model(m.input,m.get_layer('fusion_weights').output)
        a=gate.predict(make_predict_dataset(examples),verbose=0)
        assert a.shape==(8,2) and np.isfinite(a).all()
        assert np.allclose(a,0.5,atol=1e-6), (variant,a)
        print('SMOKE PASS:',variant,'params=',m.count_params(),'first gate=',a[0])
        del m,gate
        gc.collect()

## 7. Callbacks: validation-only checkpointing and gate audit

In [ ]:
class ValidationGateAudit(keras.callbacks.Callback):
    """Sampled training/validation-set gate statistics: no access to test URLs."""
    def __init__(self,variant,seed,urls):
        super().__init__()
        self.variant,self.seed=variant,seed
        self.urls=np.asarray(urls,dtype=str)
        self.records=[]
    def on_train_begin(self,logs=None):
        self.gate_model=keras.Model(self.model.input,
                                     self.model.get_layer('fusion_weights').output)
    def on_epoch_end(self,epoch,logs=None):
        a=self.gate_model.predict(make_predict_dataset(self.urls),verbose=0)
        h=-np.sum(a*np.log(np.clip(a,1e-8,1)),axis=1)/np.log(2.)
        self.records.append({
            'variant':self.variant,'Seed':self.seed,'Epoch':epoch+1,
            'Val_Macro_F1':float((logs or {}).get('val_macro_f1',np.nan)),
            'Alpha_Char_Mean':float(a[:,0].mean()),
            'Alpha_Char_SD':float(a[:,0].std()),
            'Gate_Entropy':float(h.mean()),
            'Token_gt_0p9':float((a[:,1]>0.9).mean()),
        })
        pd.DataFrame(self.records).to_csv(
            RESULT_DIR/f'gate_by_epoch_{self.variant}_seed{self.seed}.csv',index=False)


def get_callbacks(variant,seed,gate_audit):
    name=f'{variant}_seed{seed}'
    return [
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.ModelCheckpoint(
            RESULT_DIR/f'best_{name}.weights.h5',
            monitor='val_macro_f1',mode='max',save_best_only=True,
            save_weights_only=True,verbose=0),
        keras.callbacks.EarlyStopping(
            monitor='val_macro_f1',mode='max',patience=2,
            restore_best_weights=True,verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_macro_f1',mode='max',factor=0.5,
            patience=1,min_lr=1e-5,verbose=1),
        keras.callbacks.CSVLogger(RESULT_DIR/f'history_{name}.csv'),
        gate_audit,
    ]

## 8. Training grid — no test-set access until configuration is locked

In [ ]:
validation_rows=[]
val_gate_urls=val_df['url'].sample(
    n=min(2048,len(val_df)),random_state=SPLIT_SEED).astype(str).values

for variant in CANDIDATES:
    for seed in SEEDS:
        ckpt=RESULT_DIR/f'best_{variant}_seed{seed}.weights.h5'
        row_path=RESULT_DIR/f'validation_{variant}_seed{seed}.csv'
        if row_path.exists() and ckpt.exists() and not FORCE_RETRAIN:
            item=pd.read_csv(row_path).iloc[0].to_dict()
            validation_rows.append(item)
            print('REUSE:',variant,seed,'val_macro_f1=',item['Val_Macro_F1'])
            continue
        print('TRAIN:',variant,'seed',seed)
        tf.keras.backend.clear_session()
        set_all_seeds(seed)
        model=compile_model(build_model(variant))
        audit=ValidationGateAudit(variant,seed,val_gate_urls)
        history=model.fit(make_dataset(train_df,True,seed),
                          validation_data=val_ds,epochs=MAIN_EPOCHS,
                          class_weight=CLASS_WEIGHT,
                          callbacks=get_callbacks(variant,seed,audit),verbose=1)
        h=pd.DataFrame(history.history)
        best=int(h['val_macro_f1'].idxmax())
        # Audit the BEST checkpoint, not merely the last epoch.
        model.load_weights(ckpt)
        ag=keras.Model(model.input,model.get_layer('fusion_weights').output)
        aa=ag.predict(make_predict_dataset(val_gate_urls),verbose=0)
        entropy=float(np.mean(-np.sum(aa*np.log(np.clip(aa,1e-8,1)),axis=1)/np.log(2.)))
        item={'variant':variant,'Seed':seed,'Best_Epoch':best+1,
              'Epochs_Ran':len(h),'Val_Macro_F1':float(h.loc[best,'val_macro_f1']),
              'Val_Accuracy':float(h.loc[best,'val_accuracy']),
              'Gate_Entropy':entropy,'Alpha_Char_Mean':float(aa[:,0].mean()),
              'Alpha_Char_SD':float(aa[:,0].std()),
              'Token_gt_0p9':float((aa[:,1]>0.9).mean()),
              'Collapsed':bool((aa[:,0].std()<0.01) or ((aa[:,1]>0.9).mean()>0.95))}
        pd.DataFrame([item]).to_csv(row_path,index=False)
        validation_rows.append(item)
        del model,history,ag,aa,audit
        gc.collect()
validation_table=pd.DataFrame(validation_rows).sort_values(['variant','Seed'])
assert len(validation_table)==6 and not validation_table[['Val_Macro_F1']].isna().any().any()
validation_table.to_csv(RESULT_DIR/'validation_candidate_comparison.csv',index=False)
display(validation_table)

## 9. Lock one configuration using validation data **only**

The selection key is highest mean validation Macro-F1 across all three seeds. Standard deviation and worst-seed validation F1 are reported as stability diagnostics; no test-set result enters the choice. A gate-collapse indicator is reported as diagnostic rather than silently removing unfavorable runs.

In [ ]:
selection=validation_table.groupby('variant').agg(
    mean_val_macro_f1=('Val_Macro_F1','mean'),
    sd_val_macro_f1=('Val_Macro_F1','std'),
    worst_val_macro_f1=('Val_Macro_F1','min'),
    collapsed_seeds=('Collapsed','sum'),
    mean_entropy=('Gate_Entropy','mean'),
).reset_index()
selection=selection.sort_values(
    ['mean_val_macro_f1','worst_val_macro_f1'],ascending=False).reset_index(drop=True)
selected_variant=str(selection.iloc[0]['variant'])
selection['Selected']=selection['variant'].eq(selected_variant)
selection.to_csv(RESULT_DIR/'configuration_selection_VALIDATION_ONLY.csv',index=False)
display(selection)
print('LOCKED CONFIGURATION:',selected_variant)
(RESULT_DIR/'LOCKED_CONFIGURATION.txt').write_text(
    f'{selected_variant}\nSelection: highest mean validation Macro-F1, seeds 42,123,2025.\n',
    encoding='utf-8')

## 10. Single test evaluation of validation-selected configuration

In [ ]:
metric_cols=['Accuracy','Macro_Precision','Macro_Recall','Macro_F1','Weighted_F1','MCC']
selected_rows=[]
test_y=test_df['label_id'].to_numpy()
for seed in SEEDS:
    tf.keras.backend.clear_session()
    set_all_seeds(seed)
    model=compile_model(build_model(selected_variant))
    model.load_weights(RESULT_DIR/f'best_{selected_variant}_seed{seed}.weights.h5')
    probs=model.predict(make_predict_dataset(test_df['url'].astype(str).values),verbose=0)
    assert np.isfinite(probs).all() and probs.shape==(len(test_df),NUM_CLASSES)
    pred=probs.argmax(axis=1)
    pp,rr,ff,_=precision_recall_fscore_support(test_y,pred,average='macro',zero_division=0)
    row={'Model':selected_variant,'Seed':seed,'Accuracy':accuracy_score(test_y,pred),
         'Macro_Precision':pp,'Macro_Recall':rr,'Macro_F1':ff,
         'Weighted_F1':f1_score(test_y,pred,average='weighted',zero_division=0),
         'MCC':matthews_corrcoef(test_y,pred)}
    selected_rows.append(row)
    np.save(RESULT_DIR/f'selected_seed{seed}_test_predictions.npy',pred)
    pd.DataFrame(confusion_matrix(test_y,pred,labels=np.arange(NUM_CLASSES)),
                 index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(
                 RESULT_DIR/f'confusion_matrix_seed{seed}.csv')
    pd.DataFrame(classification_report(test_y,pred,target_names=CLASS_NAMES,
           output_dict=True,zero_division=0)).T.to_csv(
           RESULT_DIR/f'classification_report_seed{seed}.csv')
    gate=keras.Model(model.input,model.get_layer('fusion_weights').output)
    a=gate.predict(make_predict_dataset(test_df['url'].astype(str).values),verbose=0)
    assert np.allclose(a.sum(axis=1),1.0,atol=1e-5)
    f_entropy=-np.sum(a*np.log(np.clip(a,1e-8,1.)),axis=1)/np.log(2.)
    diag=pd.DataFrame({'true_class':[CLASS_NAMES[j] for j in test_y],
        'correct':(pred==test_y),'alpha_char':a[:,0],
        'alpha_token':a[:,1],'norm_entropy':f_entropy})
    diag.groupby(['true_class','correct'],observed=True).agg(
        n=('alpha_char','size'),char_mean=('alpha_char','mean'),
        char_sd=('alpha_char','std'),token_mean=('alpha_token','mean'),
        entropy=('norm_entropy','mean')).to_csv(
        RESULT_DIR/f'gate_by_class_error_seed{seed}.csv')
    pd.DataFrame([{'Seed':seed,'alpha_char_mean':a[:,0].mean(),
        'alpha_char_sd':a[:,0].std(),'entropy':f_entropy.mean(),
        'token_gt_0p90':(a[:,1]>0.9).mean(),
        'collapsed':((a[:,0].std()<0.01) or ((a[:,1]>0.9).mean()>0.95))}]).to_csv(
        RESULT_DIR/f'gate_diagnostic_seed{seed}.csv',index=False)
    print(seed,'Macro-F1',row['Macro_F1'],'character gate mean/sd',
          a[:,0].mean(),a[:,0].std())
    del model,gate,probs,pred,a,diag
    gc.collect()
test_results=pd.DataFrame(selected_rows)
test_results.to_csv(RESULT_DIR/'selected_stabilized_test_results.csv',index=False)
display(test_results)
summary=test_results[metric_cols].agg(['mean','std']).T
summary.to_csv(RESULT_DIR/'selected_stabilized_test_mean_std.csv')
display(summary)

## 11. Compare only with archived baseline evidence — do not mislabel as new runs

In [ ]:
# Previously executed CUDA Linux results; for descriptive historical comparison only.
prior=pd.DataFrame([
    ['Original adaptive',42,0.9763848898836645],
    ['Original adaptive',123,0.9329109456033994],
    ['Original adaptive',2025,0.9749939489204544],
    ['Fixed-average fusion',42,0.974763],
    ['Fixed-average fusion',123,0.975342],
    ['Fixed-average fusion',2025,0.976153],
],columns=['Model','Seed','Macro_F1'])
current=test_results[['Seed','Macro_F1']].copy()
current['Model']=f'Stabilized adaptive ({selected_variant})'
comparison=pd.concat([prior,current],ignore_index=True)
comparison.to_csv(RESULT_DIR/'comparison_with_archived_test_results.csv',index=False)
comparison_summary=comparison.groupby('Model').Macro_F1.agg(['mean','std','min','max'])
comparison_summary.to_csv(RESULT_DIR/'comparison_with_archived_mean_std.csv')
display(comparison_summary)
print('IMPORTANT: fixed-avg reference scores are rounded for display; use original',
      'ZIP CSV for publication-grade precision. Earlier test reuse limits claims.')

## 12. Export publication-quality figures and run readiness checks

In [ ]:
fig,ax=plt.subplots(figsize=(7.8,4.8))
for model,series in comparison.groupby('Model'):
    sub=series.sort_values('Seed')
    ax.plot(sub.Seed.astype(str),sub.Macro_F1,marker='o',label=model)
ax.set(xlabel='Random seed',ylabel='Macro-F1 (held-out test)',
       title='Adaptive fusion stabilization: predetermined random seeds')
ax.legend(fontsize=9)
ax.grid(axis='y',alpha=.22)
fig.tight_layout()
fig.savefig(RESULT_DIR/'Fig_Stabilization_ThreeSeeds.png',dpi=300)
fig.savefig(RESULT_DIR/'Fig_Stabilization_ThreeSeeds.pdf',bbox_inches='tight')
plt.show()

integrity={
   'All six training configs':len(validation_table)==len(CANDIDATES)*len(SEEDS),
   'Selection strictly uses validation':selected_variant in CANDIDATES,
   'Three selected test evaluations':len(test_results)==len(SEEDS),
   'All selected scores finite':bool(np.isfinite(test_results[metric_cols].to_numpy()).all()),
   'All checkpoints present':all((RESULT_DIR/f'best_{v}_seed{s}.weights.h5').exists()
                                  for v in CANDIDATES for s in SEEDS),
}
readiness=pd.DataFrame([{'Requirement':k,'Passed':v} for k,v in integrity.items()])
readiness.to_csv(RESULT_DIR/'experiment_readiness.csv',index=False)
display(readiness)
assert all(integrity.values())
manifest={'description':'Validation-selected gate-stabilization experiment',
 'tensorflow_version':tf.__version__,'python':sys.version.split()[0],
 'gpu':[str(x) for x in GPUS],'dataset_path':str(csv_path),
 'dataset_sha256':hashlib.sha256(Path(csv_path).read_bytes()).hexdigest(),
 'dataset_clean_n':len(df),'train_n':len(train_df),'val_n':len(val_df),
 'test_n':len(test_df),'split_seed':SPLIT_SEED,'seeds':SEEDS,
 'candidates':CANDIDATES,'selected_variant':selected_variant,
 'selection_metric':'mean validation Macro-F1',
 'test_set_warning':'Existing test set was used in previous architecture iteration.'}
(RESULT_DIR/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('DONE: output folder',RESULT_DIR)

## 13. Scientific interpretation / IEEE manuscript decision gate

1. **If stabilized adaptive beats FixedAvg mean Macro-F1 and has noncollapsed gates on all seeds**, it supports a qualified claim of better predictive performance; with only three seeds, avoid exaggerated inferential significance claims.
2. **If the scores are similar but SD falls**, state that stabilization addresses gate-collapse failure without claiming superiority.
3. **If FixedAvg remains stronger**, report FixedAvg as the strongest method; do not claim the adaptive gate is a performance improvement. Consider redesigning the proposed contribution.
4. Reporting results from the earlier test set is transparent, but because that set influenced previous architecture selection, the IEEE manuscript would benefit from an **independent external or time-split validation dataset**, and comparison of baselines under the exact same conditions.
5. Never replace the original multi-seed data or suppress a failed seed. Both archived and new data belong in the supplementary material.